# Uganda Road VCI/PCI Estimator — Colab Training

**What this notebook does:**
1. Mount Google Drive and load pre-extracted Uganda features (~500 MB)
2. Download RDD2022 road damage dataset directly from FigShare (13.3 GB)
3. Pre-train the PCI head on RDD2022 (gives accurate PCI estimates)
4. Train all three heads (Defect + vVCI + PCI) on Uganda features
5. Save the final checkpoint to Google Drive

**Before running:** Upload these two files to your Google Drive:
- `outputs/features.npz` (from `scripts/extract_features.py`)
- `outputs/dataset.csv` (from `scripts/prepare_data.py`)

**Runtime:** GPU → T4 or better. Expected time: ~30 min.

In [ ]:
# ── 0. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('VRAM (GB)  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: no GPU detected — change Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# ── EDIT THESE PATHS to match where you put the files in your Drive ──────────
DRIVE_ROOT    = '/content/drive/MyDrive/vci_estimator'
FEATURES_PATH = f'{DRIVE_ROOT}/features.npz'
DATASET_PATH  = f'{DRIVE_ROOT}/dataset.csv'
OUTPUT_DIR    = f'{DRIVE_ROOT}/outputs'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

assert os.path.exists(FEATURES_PATH), f'features.npz not found at {FEATURES_PATH}'
assert os.path.exists(DATASET_PATH),  f'dataset.csv not found at {DATASET_PATH}'
print('Drive mounted. Files found.')

In [ ]:
# ── 2. Install dependencies + clone project scripts ──────────────────────────
!pip install timm -q

# Copy project code from Drive (upload the whole project zip, or paste scripts inline)
# Option A: project zip in Drive
PROJECT_ZIP = f'{DRIVE_ROOT}/vci_estimator_scripts.zip'
if os.path.exists(PROJECT_ZIP):
    !unzip -q {PROJECT_ZIP} -d /content/project
    import sys; sys.path.insert(0, '/content/project')
    print('Project scripts loaded from zip.')
else:
    print('No zip found — will use inline implementations below.')

import numpy as np, pandas as pd
print('numpy:', np.__version__, '| pandas:', pd.__version__)

In [ ]:
# ── 3. Inspect features + dataset ────────────────────────────────────────────
import numpy as np, pandas as pd

npz = np.load(FEATURES_PATH, allow_pickle=True)
feats = npz['features']   # (N, 1536) float16
paths = npz['image_paths']
print(f'Features shape : {feats.shape}  dtype={feats.dtype}')
print(f'File size      : {os.path.getsize(FEATURES_PATH)/1e6:.0f} MB')

df = pd.read_csv(DATASET_PATH)
print(f'\nDataset rows   : {len(df):,}')
print(f'Splits         : {df["split"].value_counts().to_dict()}')
print(f'Roads          : {df["road_name"].nunique()}')
print(f'Survey years   : {sorted(df["survey_year"].dropna().unique().astype(int).tolist())}')
print(f'vVCI range     : {df["vvci"].min():.1f} – {df["vvci"].max():.1f}  mean={df["vvci"].mean():.1f}')

In [ ]:
# ── 3b. Dataset overview visualizations ──────────────────────────────────────
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import numpy as np, pandas as pd, os

df = pd.read_csv(DATASET_PATH)
DEFECT_COLS_ALL  = ['all_cracking_grade','wide_cracking_grade','ravelling_grade',
                    'bleeding_grade','drainage_road_grade','pothole_grade']
DEFECT_LABELS    = ['All\nCracking','Wide\nCracking','Ravelling','Bleeding','Drainage\n(road)','Potholes']
GRADE_COLOURS_5  = ['#22c55e','#84cc16','#eab308','#f97316','#ef4444']

fig = plt.figure(figsize=(18, 12))
fig.suptitle('Uganda Road Survey — Dataset Overview', fontsize=16, fontweight='bold', y=0.99)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── (A) vVCI distribution ──
ax_vvci = fig.add_subplot(gs[0, 0])
ax_vvci.hist(df['vvci'], bins=30, color='#3b82f6', alpha=0.85, edgecolor='white')
for thresh, lbl, col in [(40,'Poor','#ef4444'),(60,'Fair','#f97316'),(80,'Good','#22c55e')]:
    ax_vvci.axvline(thresh, color=col, linestyle='--', lw=1.5, alpha=0.8, label=lbl)
ax_vvci.set_title('vVCI Distribution', fontweight='bold')
ax_vvci.set_xlabel('vVCI (0–100)'); ax_vvci.set_ylabel('Images')
ax_vvci.legend(fontsize=8)

# ── (B) Segments per split ──
ax_split = fig.add_subplot(gs[0, 1])
seg_df   = df.drop_duplicates(subset=['road_code','segment_start','segment_end','survey_year'])
splits   = seg_df['split'].value_counts().reindex(['train','val','test'], fill_value=0)
split_c  = ['#3b82f6','#f59e0b','#ef4444']
bars     = ax_split.bar(splits.index, splits.values, color=split_c, alpha=0.85)
for bar, v in zip(bars, splits.values):
    ax_split.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(v),
                  ha='center', fontweight='bold', fontsize=11)
ax_split.set_title('Segments per Split\n(70 / 15 / 15 random)', fontweight='bold')
ax_split.set_xlabel('Split'); ax_split.set_ylabel('Unique Segments')

# ── (C) Images per road × year ──
ax_ry = fig.add_subplot(gs[0, 2])
ry = df.groupby(['road_name','survey_year']).size().reset_index(name='n')
ry_labels = [f"{r}  {int(y)}" for r,y in zip(ry['road_name'], ry['survey_year'])]
ry_cols   = ['#6366f1','#8b5cf6','#a78bfa','#0ea5e9','#38bdf8','#7dd3fc'][:len(ry)]
hbars     = ax_ry.barh(range(len(ry)), ry['n']/1000, color=ry_cols, alpha=0.85)
ax_ry.set_yticks(range(len(ry))); ax_ry.set_yticklabels(ry_labels, fontsize=9)
ax_ry.set_xlabel('Images (thousands)'); ax_ry.set_title('Images per Road × Year', fontweight='bold')
for hb, v in zip(hbars, ry['n']):
    ax_ry.text(v/1000 + 0.4, hb.get_y() + hb.get_height()/2,
               f'{v/1000:.1f}k', va='center', fontsize=8)

# ── (D–I) Defect grade distributions ──
ax_def = fig.add_subplot(gs[1, :])
x = np.arange(len(DEFECT_COLS_ALL))
w = 0.15
for gi in range(5):
    counts = [(df[c] == gi+1).sum() if c in df.columns else 0 for c in DEFECT_COLS_ALL]
    ax_def.bar(x + gi*w - 0.30, counts, w, label=f'Grade {gi+1}',
               color=GRADE_COLOURS_5[gi], alpha=0.85, edgecolor='white')
ax_def.set_xticks(x); ax_def.set_xticklabels(DEFECT_LABELS)
ax_def.set_title('Defect Grade Distributions', fontweight='bold')
ax_def.set_xlabel('Defect Type'); ax_def.set_ylabel('Images')
ax_def.legend(loc='upper right', fontsize=9)

plt.savefig(f'{OUTPUT_DIR}/dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Dataset overview saved → {OUTPUT_DIR}/dataset_overview.png')

In [ ]:
# ── 4. Download RDD2022 from FigShare ─────────────────────────────────────────
# RDD2022 is publicly available. We use Japan subset (highest quality annotations).
# Full dataset is 13.3 GB; Japan alone is ~3 GB — enough for PCI pre-training.
import os

RDD_DIR = '/content/rdd2022'
os.makedirs(RDD_DIR, exist_ok=True)

# Check if already downloaded to Drive (saves re-downloading)
RDD_DRIVE_CACHE = f'{DRIVE_ROOT}/rdd2022_Japan.zip'

if os.path.exists(RDD_DRIVE_CACHE):
    print('Using cached RDD2022 Japan from Drive ...')
    !unzip -q {RDD_DRIVE_CACHE} -d {RDD_DIR}
else:
    print('Downloading RDD2022 Japan from FigShare (~3 GB) ...')
    # FigShare DOI: 10.6084/m9.figshare.21504930
    # Japan subset URL (direct download)
    JAPAN_URL = 'https://figshare.com/ndownloader/files/38346750'
    !wget -q --show-progress -O /content/rdd2022_Japan.zip "{JAPAN_URL}"
    !unzip -q /content/rdd2022_Japan.zip -d {RDD_DIR}
    # Cache to Drive so we don't re-download next session
    !cp /content/rdd2022_Japan.zip {RDD_DRIVE_CACHE}
    print('Cached to Drive for future sessions.')

# Verify
import glob
japan_imgs = glob.glob(f'{RDD_DIR}/**/images/*.jpg', recursive=True)
japan_xmls = glob.glob(f'{RDD_DIR}/**/annotations/xml/*.xml', recursive=True)
print(f'RDD2022 Japan: {len(japan_imgs):,} images, {len(japan_xmls):,} annotations')

In [ ]:
# ── 5. Pre-train PCI head on RDD2022 ─────────────────────────────────────────
# The PCI head sees international road damage images with ASTM-derived PCI labels
# before it ever sees Uganda data. This gives it a much better initialisation.

PCI_PRETRAIN_PATH = f'{OUTPUT_DIR}/pci_head.pt'

if os.path.exists(PCI_PRETRAIN_PATH):
    print(f'Pre-trained PCI head found at {PCI_PRETRAIN_PATH} — skipping pre-training.')
    print('Delete the file and re-run this cell to force re-training.')
else:
    !python /content/project/scripts/pretrain_pci.py \
        --data        {RDD_DIR} \
        --countries   Japan \
        --output-dir  {OUTPUT_DIR}/pci_pretrain \
        --device      cuda \
        --epochs-stage1 10 \
        --epochs-stage2 10 \
        --batch-size  64
    
    import shutil
    shutil.copy(f'{OUTPUT_DIR}/pci_pretrain/pci_head.pt', PCI_PRETRAIN_PATH)
    print(f'Pre-trained PCI head saved → {PCI_PRETRAIN_PATH}')

In [ ]:
# ── 6. Train all heads on Uganda features ────────────────────────────────────
# --resplit: random 70/15/15 by segment → 408 training segs (vs 231 with temporal split)
# --lambda-consist 0.3: consistency loss anchors vVCI head to the known MoWT formula
# --batch-size 512: no backbone pass, so large batches fit easily on T4 VRAM
# Training 60 epochs takes ~20–35 min on T4.

!python /content/project/scripts/train_features.py \
    --features        {FEATURES_PATH} \
    --dataset         {DATASET_PATH} \
    --output-dir      {OUTPUT_DIR} \
    --device          cuda \
    --epochs          60 \
    --batch-size      512 \
    --lambda-consist  0.3 \
    --resplit \
    --pci-pretrain    {PCI_PRETRAIN_PATH}

In [ ]:
# ── 8. Post-training analysis: per-defect accuracy + vVCI scatter ─────────────
import torch, torch.nn.functional as F
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import sys, os

sys.path.insert(0, '/content/project')
from src.models.model import HeadsModel
from src.data.dataset import DEFECT_COLS

DEFECT_NAMES_SHORT = ['All Cracking','Wide Cracking','Ravelling','Bleeding','Drainage','Potholes']
GRADE_COLOURS_6    = ['#3b82f6','#8b5cf6','#22c55e','#f59e0b','#f97316','#ef4444']
GRADE_COLOURS_5    = ['#22c55e','#84cc16','#eab308','#f97316','#ef4444']

ckpt_path = f'{OUTPUT_DIR}/checkpoints/best.pt'
if not os.path.exists(ckpt_path):
    print('best.pt not found — run training cell first.')
else:
    # ── Load model ──────────────────────────────────────────────────────────
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model = HeadsModel(); model.load_state_dict(ckpt['model']); model.eval()

    # ── Load features + dataset ─────────────────────────────────────────────
    npz   = np.load(FEATURES_PATH, allow_pickle=True)
    feats = npz['features'].astype(np.float32)
    paths = npz['image_paths']
    df    = pd.read_csv(DATASET_PATH)
    path_to_feat = {p: i for i, p in enumerate(paths)}

    # ── Evaluate on test segments ────────────────────────────────────────────
    test_df = df[df['split'] == 'test']
    groups  = test_df.groupby(['road_code','segment_start','segment_end','survey_year'])

    all_true  = [[] for _ in DEFECT_COLS]
    all_pred  = [[] for _ in DEFECT_COLS]
    true_vvci, pred_vvci = [], []

    with torch.no_grad():
        for _, grp in groups:
            fidxs = [path_to_feat[p] for p in grp['image_path'] if p in path_to_feat]
            if not fidxs: continue
            feat = torch.tensor(feats[fidxs]).mean(0, keepdim=True)
            logits_list = model.defect_head(feat)
            true_vvci.append(float(grp['vvci'].iloc[0]))
            pred_vvci.append(float(model.vvci_head(feat).squeeze()))
            for i, col in enumerate(DEFECT_COLS):
                all_true[i].append(int(grp[col].iloc[0]))
                all_pred[i].append(int(F.softmax(logits_list[i], dim=-1).argmax().item()) + 1)

    # ── Metrics ─────────────────────────────────────────────────────────────
    accs    = [np.mean(np.array(all_true[i]) == np.array(all_pred[i])) for i in range(len(DEFECT_COLS))]
    within1 = [np.mean(np.abs(np.array(all_true[i])-np.array(all_pred[i])) <= 1) for i in range(len(DEFECT_COLS))]
    tv = np.array(true_vvci); pv = np.array(pred_vvci); err = pv - tv
    mae = np.mean(np.abs(err)); rmse = np.sqrt(np.mean(err**2))
    r, _ = stats.pearsonr(tv, pv)

    # ════════════════════════════════════════════════════════════════════════
    # Figure 1 — Per-defect accuracy
    # ════════════════════════════════════════════════════════════════════════
    fig1, axes1 = plt.subplots(1, 2, figsize=(16, 5))
    fig1.suptitle('Per-Defect Accuracy (Test Set)', fontsize=14, fontweight='bold')

    bars1 = axes1[0].barh(DEFECT_NAMES_SHORT, [a*100 for a in accs], color=GRADE_COLOURS_6, alpha=0.85)
    axes1[0].axvline(20, color='gray', linestyle='--', alpha=0.5, label='Random baseline (20%)')
    axes1[0].set_xlabel('Accuracy (%)'); axes1[0].set_title('Exact Grade Accuracy')
    axes1[0].set_xlim(0, 105); axes1[0].legend()
    for bar, a in zip(bars1, accs):
        axes1[0].text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
                      f'{a*100:.1f}%', va='center', fontsize=10, fontweight='bold')

    bars2 = axes1[1].barh(DEFECT_NAMES_SHORT, [w*100 for w in within1], color=GRADE_COLOURS_6, alpha=0.85)
    axes1[1].axvline(50, color='gray', linestyle='--', alpha=0.5, label='50% ref')
    axes1[1].set_xlabel('Within-1 Accuracy (%)'); axes1[1].set_title('Within-1 Grade Accuracy')
    axes1[1].set_xlim(0, 105); axes1[1].legend()
    for bar, w in zip(bars2, within1):
        axes1[1].text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
                      f'{w*100:.1f}%', va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/per_defect_accuracy.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # Figure 2 — vVCI scatter + error analysis
    # ════════════════════════════════════════════════════════════════════════
    fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
    fig2.suptitle('vVCI Prediction Analysis (Test Set)', fontsize=14, fontweight='bold')

    # Colour points by true condition band
    band_c = ['#22c55e' if v>80 else '#84cc16' if v>60 else '#f97316' if v>40 else '#ef4444' for v in tv]
    mn, mx = min(tv.min(), pv.min())-2, max(tv.max(), pv.max())+2

    axes2[0].scatter(tv, pv, c=band_c, alpha=0.7, s=65, edgecolors='white', lw=0.4)
    axes2[0].plot([mn,mx],[mn,mx],'k--',alpha=0.4,label='Perfect')
    axes2[0].plot([mn,mx],[mn+mae,mx+mae],'r:',alpha=0.5)
    axes2[0].plot([mn,mx],[mn-mae,mx-mae],'r:',alpha=0.5,label=f'±MAE={mae:.1f}')
    axes2[0].set_xlabel('True vVCI'); axes2[0].set_ylabel('Predicted vVCI')
    axes2[0].set_title(f'Scatter  r={r:.3f}  MAE={mae:.1f}  RMSE={rmse:.1f}')
    from matplotlib.patches import Patch
    axes2[0].legend(handles=[
        Patch(color='#22c55e',label='Good >80'), Patch(color='#84cc16',label='Fair 60–80'),
        Patch(color='#f97316',label='Poor 40–60'), Patch(color='#ef4444',label='Bad <40')
    ], fontsize=7, loc='upper left')

    axes2[1].hist(err, bins=25, color='#3b82f6', alpha=0.85, edgecolor='white')
    axes2[1].axvline(0, color='k', linestyle='--', alpha=0.5, label='Zero error')
    axes2[1].axvline(mae,  color='red', linestyle='--', alpha=0.7, label=f'MAE={mae:.1f}')
    axes2[1].axvline(-mae, color='red', linestyle='--', alpha=0.7)
    axes2[1].set_xlabel('Prediction error (vVCI points)'); axes2[1].set_ylabel('Segments')
    axes2[1].set_title('Error Distribution'); axes2[1].legend(fontsize=8)

    bands      = ['Bad\n(<40)','Poor\n(40–60)','Fair\n(60–80)','Good\n(>80)']
    band_masks = [tv<40,(tv>=40)&(tv<60),(tv>=60)&(tv<80),tv>=80]
    band_maes  = [np.mean(np.abs(err[m])) if m.sum()>0 else 0 for m in band_masks]
    band_ns    = [m.sum() for m in band_masks]
    bars3      = axes2[2].bar(bands, band_maes, color=['#ef4444','#f97316','#84cc16','#22c55e'], alpha=0.85)
    for bar, bv, n in zip(bars3, band_maes, band_ns):
        axes2[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                      f'{bv:.1f}\n(n={n})', ha='center', fontsize=9, fontweight='bold')
    axes2[2].set_ylabel('MAE (vVCI points)'); axes2[2].set_title('MAE per Condition Band')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/vvci_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── Summary table ────────────────────────────────────────────────────────
    print(f'\n{"═"*48}')
    print(f'  TEST SET RESULTS — Uganda Road VCI Estimator')
    print(f'{"─"*48}')
    print(f'  Segments evaluated      : {len(tv)}')
    print(f'  MAE  (vVCI)             : {mae:.2f} points')
    print(f'  RMSE (vVCI)             : {rmse:.2f} points')
    print(f'  Pearson r               : {r:.3f}')
    print(f'{"─"*48}')
    print(f'  Mean exact defect acc   : {np.mean(accs)*100:.1f}%  (random=20%)')
    print(f'  Mean within-1 defect acc: {np.mean(within1)*100:.1f}%')
    print(f'{"─"*48}')
    for name, a, w in zip(DEFECT_NAMES_SHORT, accs, within1):
        print(f'  {name:<20} acc={a*100:.0f}%  ±1={w*100:.0f}%')
    print(f'{"═"*48}')
    print(f'\nPlots saved to {OUTPUT_DIR}/')

In [ ]:
# ── 7. Training curves ────────────────────────────────────────────────────────
import os, pandas as pd, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

metrics_path = f'{OUTPUT_DIR}/metrics_features.csv'
if not os.path.exists(metrics_path):
    print(f'metrics_features.csv not found at {metrics_path}')
    print('Run the training cell first.')
else:
    m = pd.read_csv(metrics_path)
    best_idx = int(m['val_mae_vvci'].idxmin())

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle('Training Progress — Uganda Road VCI Estimator', fontsize=15, fontweight='bold', y=0.98)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

    # ── (1) Total training loss ──
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(m['epoch'], m['train_loss'], color='#3b82f6', lw=2.5, label='Train loss')
    ax1.set_title('Total Training Loss', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.grid(alpha=0.3); ax1.legend()

    # ── (2) Validation MAE (vVCI) ──
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(m['epoch'], m['val_mae_vvci'], color='#f97316', lw=2.5, label='Val MAE')
    best_mae = m['val_mae_vvci'][best_idx]
    best_ep  = m['epoch'][best_idx]
    ax2.axvline(best_ep, color='red', linestyle='--', alpha=0.65, label=f'Best (ep {best_ep})')
    ax2.scatter([best_ep], [best_mae], color='red', s=90, zorder=5)
    ax2.annotate(f'  MAE={best_mae:.2f}', xy=(best_ep, best_mae),
                 xytext=(4, 4), textcoords='offset points', fontsize=9, color='red')
    ax2.set_title('Validation MAE (vVCI)', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE (vVCI points)')
    ax2.grid(alpha=0.3); ax2.legend()

    # ── (3) Validation RMSE (vVCI) ──
    ax3 = fig.add_subplot(gs[1, 0])
    if 'val_rmse_vvci' in m.columns:
        ax3.plot(m['epoch'], m['val_rmse_vvci'], color='#8b5cf6', lw=2.5, label='Val RMSE')
        ax3.scatter([best_ep], [m['val_rmse_vvci'][best_idx]], color='#8b5cf6', s=90, zorder=5)
        ax3.axvline(best_ep, color='red', linestyle='--', alpha=0.4)
        ax3.set_title('Validation RMSE (vVCI)', fontweight='bold')
        ax3.set_xlabel('Epoch'); ax3.set_ylabel('RMSE (vVCI points)')
        ax3.grid(alpha=0.3); ax3.legend()

    # ── (4) Defect classification accuracy ──
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.plot(m['epoch'], m['val_acc_defect_mean'] * 100, color='#22c55e', lw=2.5, label='Mean defect acc')
    ax4.axhline(20, color='gray', linestyle='--', alpha=0.5, label='Random baseline (20%)')
    ax4.set_title('Defect Grade Accuracy (5-class mean)', fontweight='bold')
    ax4.set_xlabel('Epoch'); ax4.set_ylabel('Accuracy (%)')
    ax4.grid(alpha=0.3); ax4.legend()

    plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Summary
    best_rmse = m['val_rmse_vvci'][best_idx] if 'val_rmse_vvci' in m.columns else float('nan')
    print(f'\n{"═"*42}')
    print(f'  Best epoch       : {best_ep}')
    print(f'  Best val MAE     : {best_mae:.2f} vVCI points')
    print(f'  Val RMSE         : {best_rmse:.2f} vVCI points')
    print(f'  Final defect acc : {m["val_acc_defect_mean"].iloc[-1]*100:.1f}%')
    print(f'{"═"*42}')
    print(f'\nPlot saved → {OUTPUT_DIR}/training_curves.png')

In [ ]:
# ── 8. Verify checkpoint + print final metrics ────────────────────────────────
import torch

ckpt_path = f'{OUTPUT_DIR}/checkpoints/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print(f'Checkpoint epoch  : {ckpt["epoch"]}')
    print(f'Best val MAE      : {ckpt["val_mae"]:.2f}')
    print(f'Feature-based     : {ckpt.get("feature_based", False)}')
    print(f'Checkpoint size   : {os.path.getsize(ckpt_path)/1e6:.1f} MB')
    print(f'\nSaved to Drive at : {ckpt_path}')
else:
    print('best.pt not found')

## After training

Download `best.pt` from your Drive and place it at `outputs/checkpoints/best.pt` on your local machine.

**Note**: Because training used pre-extracted features (no backbone), `best.pt` contains only the three head weights. To run the full inference pipeline (which needs the backbone), the `api/inference.py` must be updated to either:
1. Load the full EfficientNet-B3 backbone + attach the trained head weights
2. Or export using `scripts/export_features_model.py` (to be written)

The `evaluate.py` script will work as-is if you point it at `dataset.csv` and use the feature-based inference path.